# Complex Numbers, Wirtinger Derivatives, and Signals

This notebook supports the Part 1 article. It focuses on visuals while calling reusable Python modules for the core math and data generation.

In [1]:
import numpy as np
from IPython.display import Markdown, display

from complex_core import CONSTELLATIONS, from_dxdy, conj_from_dxdy
from complex_plots import (
    format_wirtinger_steps_markdown,
    modulation_notes_markdown,
    plot_backprop_training,
    plot_constellation_rotation_grid,
    plot_descent_trajectory_comparison,
    plot_iq_map_geometry,
    plot_iq_sample_efficiency,
    plot_rotation_vectors,
    plot_sinusoid_and_iq,
)

## 1) Imaginary numbers as rotation

Multiplying by `i` rotates a point by 90 degrees. Multiplying by `e^(i*theta)` rotates by any angle.

In [2]:
fig = plot_rotation_vectors(x=1.0, y=1.0, angle_deg=45.0)
fig.show()

## 2) Real derivatives vs Wirtinger derivatives

For `z = x + iy`, Wirtinger operators are:

- `d/dz  = 1/2 (d/dx - i d/dy)`
- `d/dz* = 1/2 (d/dx + i d/dy)`

In [3]:
x, y = 1.2, -0.7
df_dx, df_dy = 2 * x, 2 * y  # for f(z)=|z|^2
print('Real partials:')
print('  df/dx =', df_dx)
print('  df/dy =', df_dy)
print('Wirtinger:')
print('  df/dz  =', from_dxdy(df_dx, df_dy))
print('  df/dz* =', conj_from_dxdy(df_dx, df_dy))

Real partials:
  df/dx = 2.4
  df/dy = -1.4
Wirtinger:
  df/dz  = (1.2+0.7j)
  df/dz* = (1.2-0.7j)


In [4]:
display(Markdown(format_wirtinger_steps_markdown('abs2', x=1.2, y=-0.7)))

**Given:** `z = 1.200 + (-0.700)i = 1.200-0.700j`

**Function:** `f(z) = |z|^2 = x^2 + y^2`

**Step 1:** `f(z) = 1.9300`

**Step 2:** `df/dx = 2.4000`, `df/dy = -1.4000`

**Step 3:**
`df/dz  = 0.5*(df/dx - i*df/dy) = 1.2000+0.7000j`

`df/dz* = 0.5*(df/dx + i*df/dy) = 1.2000-0.7000j`


## 3) Why Wirtinger matters for IQ backprop

For a real-valued loss `L` with complex parameters `z`, the steepest-descent update is:

`z <- z - lr * dL/dz*`

This section shows three things:

1. Correctness: `dL/dz*` and split real/imag updates are equivalent; using `dL/dz` is the wrong update direction.
2. Geometry: complex maps naturally commute with IQ rotation, while generic real 2x2 maps need not.
3. Practice: with narrow phase coverage and limited data, complex structure can improve sample efficiency.

Framework note from current docs:
- PyTorch complex autograd returns the conjugate Wirtinger gradient (`dL/dz*`) for real losses.
- TensorFlow uses the same convention.
- JAX uses a different convention for some complex differentiation APIs; for complex-parameter optimization with real losses you typically conjugate its gradient before updating.

In [5]:
fig, summary, stats = plot_descent_trajectory_comparison(target=1.0 + 2.0j, w_init=-2.0 + 1.5j, lr=0.25, epochs=20)
fig.show()
print(summary)

Final losses -- Wirtinger: 9.302e-05, Split-real: 9.302e-05, Wrong dL/dz: 1.881e+03


### Why split-real works and why `dL/dz` fails

Use the toy loss `L(w)=|w-a|^2` with `w=u+iv` and `a=alpha+i*beta`.

- Real partials are `dL/du = 2(u-alpha)` and `dL/dv = 2(v-beta)`.
- Split-real gradient descent updates
  `u <- u - (lr/2) dL/du`, `v <- v - (lr/2) dL/dv`,
  so `w <- w - lr[(u-alpha) + i(v-beta)] = w - lr (w-a)`.
- But `(w-a)` is exactly `dL/dw*`, so split-real and Wirtinger-conjugate are the same update.

Why the wrong red update fails:

- `dL/dw = conj(w-a)`, so wrong update is `w <- w - lr * conj(w-a)`.
- If error `e=w-a=x+iy`, then `e_next = (1-lr)x + i(1+lr)y`.
- Real error shrinks, but imaginary error grows, so loss eventually increases.

### 3a) IQ geometry check (what this plot means)

This section tests one question: **if we rotate all IQ samples first, then map them, do we get the same result as mapping first, then rotating?**

For a map `f`, we compare `f(Rx)` with `R(fx)` where `R` is a global phase rotation.

How to read the figure:

- Panel 1 shows the original burst `x` and its rotated copy `R(x)`.
- Panel 2 (complex map) overlays `w(Rx)` and `R(wx)`. They should overlap almost perfectly.
- Panel 3 (unconstrained real 2x2 map) shows `A(Rx)` vs `R(Ax)`. They usually separate.

Takeaway: complex multiplication naturally respects IQ global phase rotation, while a generic real 2x2 map does not unless constrained to complex form.

In [6]:
fig, summary, stats = plot_iq_map_geometry(seed=4, n_symbols=256, phase_deg=30.0, delta_deg=45.0)
fig.show()
print(summary)

Commutation error mean ||f(Rx)-R(fx)||^2: complex=1.60e-15, unconstrained real 2x2=7.62e-02, constrained-real(complex form)=1.40e-15. Read the plot left-to-right: panel 2 overlaps (good), panel 3 separates (not rotation-equivariant).


### 3b) When to use complex+Wirtinger: phase nuisance with limited data

Both formulations can optimize the same real objective.
The difference below is inductive bias: a complex-structured map (rotation+scale form) versus an unconstrained real 2x2 map.
This demonstrates when structure helps, not that real calculus is incorrect.

In [7]:
fig, summary, stats = plot_iq_sample_efficiency(n_train=24, train_phase_deg=15.0, n_test=1024, noise=0.15, seeds=tuple(range(12)), lr=0.3, epochs=35)
fig.show()
print(summary)

Train MSE -- complex: 0.0397, real: 0.0388. Mean test error gap (real - complex): 0.0011. This gap comes from useful structure constraints, not a different objective (complex: 2 real dof, real 2x2: 4 real dof).


## 4) Complex numbers in signals

A complex sinusoid bundles amplitude and phase naturally. A phase shift rotates the whole signal in the IQ plane.

In [8]:
fig = plot_sinusoid_and_iq(amplitude=1.0, frequency=2.0, phase_deg=0.0, snr_db=30.0, shift_deg=45.0, seed=0)
fig.show()

## 5) Data preview for Part 2

First we show all classes with no rotation to establish the baseline geometry. Then we apply global rotations and show that the class identity stays the same while the constellation orientation changes.

In [9]:
print('Available constellations:', list(CONSTELLATIONS))
fig = plot_constellation_rotation_grid(rotations_deg=(0.0,), snr_db=20.0, n_symbols=256, seed=0)
fig.show()
display(Markdown(modulation_notes_markdown()))

Available constellations: ['bpsk', 'qpsk', '8psk', '16qam']


### Signal classes used in Part 2

- **BPSK**: Binary phase-shift keying with 2 phase states (1 bit/symbol). Robust low-rate links like telemetry, satellite control, and deep-space style channels.
- **QPSK**: Quadrature phase-shift keying with 4 phase states (2 bits/symbol). Widely used in cellular, satellite, and many digital radio systems.
- **8PSK**: Phase-shift keying with 8 phase states (3 bits/symbol). Used when higher spectral efficiency is needed, for example in some satellite broadcast links.
- **16QAM**: Quadrature amplitude modulation with 16 amplitude-phase points (4 bits/symbol). Common in Wi-Fi, LTE/5G, cable modems, and other high-throughput links.

In [10]:
fig = plot_constellation_rotation_grid(rotations_deg=(0.0, 45.0, 90.0), snr_db=20.0, n_symbols=256, seed=11)
fig.show()